## Do practice improvements predict reductions in vulnerability? 

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import seaborn as sns
from settings import load_settings
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from matplotlib import pyplot as plt
from capstone import construct_final_dataset as fd

In [ ]:
settings = load_settings()

In [ ]:
df = pl.read_parquet(settings.research_question_1_dataset_path)

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(10, 6))
numeric_df = df.select(cs.numeric())
corr_df = numeric_df.drop_nulls().corr().to_pandas()
sns.heatmap(corr_df, annot=True, yticklabels=corr_df.columns, xticklabels=corr_df.columns, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix (vulnerability features)")
plt.show()

In [ ]:
features = [
    fd.C_PACKAGE_DEPENDENCY_COUNT,
    fd.C_PACKAGE_TOTAL_DOWNLOADS,
    fd.C_REPOSITORY_AGE_IN_YEARS,
    fd.C_REPOSITORY_COMMIT_STALENESS_IN_DAYS,
    fd.C_REPOSITORY_CONTRIBUTIONS_COUNT,
    fd.C_REPOSITORY_SIZE_IN_KB, 
    fd.P_AGGREGATED_SCORE
]

X = df[features]
y = df[fd.T_VULNERABILITY_COUNT]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
combined = X_train.with_columns(target=y_train)
cleaned = combined.filter(pl.col("target").is_not_null())
X_train_clean = cleaned.drop("target")
y_train_clean = cleaned.select("target").to_series()
log_scaler = FunctionTransformer(np.log1p, validate=True)

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', log_scaler),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train_clean, y_train_clean, cv=kf, scoring='neg_mean_absolute_error')
print(f"Training Cross-Validation MAE: {-cv_scores.mean():.4f}")
pipeline.fit(X_train_clean, y_train_clean)
print("R2 Score on Training Set:", r2_score(y_train_clean, pipeline.predict(X_train_clean)))

In [ ]:
test_combined = X_test.with_columns(target=y_test)
test_cleaned = test_combined.filter(pl.col("target").is_not_null())
X_test_clean = test_cleaned.drop("target")
y_test_clean = test_cleaned.select("target").to_series()

pipeline.fit(X_train_clean.to_numpy(), y_train_clean.to_numpy())
y_pred = pipeline.predict(X_test_clean.to_numpy())
final_mae = mean_absolute_error(y_test_clean, y_pred)
print(f"Final Test MAE: {final_mae:.4f}")
r2_score = r2_score(y_test_clean, y_pred)
print(f"R2 Score on Test Set: {r2_score:.4f}")

In [ ]:
rf_model = pipeline.named_steps['model']
feature_names = X_train_clean.columns
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importances['feature'], importances['importance'])
plt.gca().invert_yaxis()  # most important at top
plt.xlabel('Importance Score')
plt.title('Top Predictors in Random Forest Model')
plt.show()